In [ ]:
import torch
import torchvision
import torch.nn as nn
import math

import numpy as np
from einops.layers.torch import Rearrange

import os
import shutil
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd

from collections import Counter 
from typing import *

import ray
from ray import tune
from ray.tune.schedulers import ASHAScheduler
from ray.tune import Checkpoint
import tempfile

import matplotlib.pyplot as plt

# Homework description
 
In this homework, you have to build an image classification model that treats each image as a sequence of sub-images. Very similar to the recent architecture of [Visual Transformer](https://arxiv.org/pdf/2010.11929.pdf). 

In [ ]:
dataset = torchvision.datasets.Caltech256('data/')

targets = [label for _, label in dataset]
class_counts = Counter(targets)
num_classes = len(class_counts)
total_samples = len(targets)

weights = torch.tensor([total_samples / class_counts[i] for i in range(num_classes)])

def split_caltech256(dataset_path, output_path, train_ratio=0.7, val_ratio=0.15, random_state=42):
    if not os.path.exists(output_path):
        os.makedirs(output_path, exist_ok=True)

    dataset_path, output_path = Path(dataset_path), Path(output_path)
    
    for category in [d for d in dataset_path.iterdir() if d.is_dir()]:
        images = list(category.glob('*.[jJ][pP][gG]')) + list(category.glob('*.[jJ][pP][eE][gG]'))
        if not images: continue
        
        train, temp = train_test_split(images, train_size=train_ratio, random_state=random_state)
        val, test = train_test_split(temp, train_size=val_ratio/(1-train_ratio), random_state=random_state)
        
        for split, imgs in [('train', train), ('val', val), ('test', test)]:
            dest = output_path / split / category.name
            dest.mkdir(parents=True, exist_ok=True)
            for img in imgs: shutil.copy2(img, dest / img.name)

split_caltech256(dataset_path='data/caltech256/256_ObjectCategories', output_path='data/caltech256_splitted')

In [ ]:
def load_data(config, train_transform, test_transform):
    train_dataset = torchvision.datasets.ImageFolder('data/caltech256_splitted/train', transform=train_transform)
    valid_dataset = torchvision.datasets.ImageFolder('data/caltech256_splitted/val', transform=test_transform)
    test_dateset = torchvision.datasets.ImageFolder('data/caltech256_splitted/test', transform=test_transform)

    train_loader = torch.utils.data.DataLoader(
        train_dataset, 
        batch_size=int(config['batch_size']),
        shuffle=True, 
        num_workers=4,
        pin_memory=True, 
        persistent_workers=True
    )
    valid_loader = torch.utils.data.DataLoader(
        valid_dataset, 
        batch_size=int(config['batch_size']),
        shuffle=False, 
        num_workers=4,
        pin_memory=True, 
        persistent_workers=True
    )
    test_loader = torch.utils.data.DataLoader(
        test_dateset, 
        batch_size=int(config['batch_size']),
        shuffle=False, 
        num_workers=4,
        pin_memory=True, 
        persistent_workers=True
    )

    return train_loader, valid_loader, test_loader, train_dataset.classes

In [ ]:
# You can check how PatchEmbeddings class splits the image into patches using the image_to_patches function

class PatchEmbeddings(torch.nn.Module):
    def __init__(self, patch_size, d_model, channels=3):
        """ Patch Embedding class. 
                Takes image as an input, splits it into a number of patches of the pre-determined size (patch_size) 
                and applies a linear transformation to convert the patch into a vector representation.

            Arguments: 
                patch_size: int
                    size of the patch to cut from an image
                d_model: int
                    linear representation size of the patch
                channels: int
                    number of channels in the input image
        """
        super().__init__()
        self.patch_size = patch_size
        self.channels = channels
        self.d_model = d_model
        self.patch_dim = channels * patch_size * patch_size

        self.patch_embeddings = Rearrange(
            'b c (h p1) (w p2) -> b (h w) (p1 p2 c)',
            p1=patch_size, p2=patch_size
        )
        self.embedding = nn.Linear(
            in_features=self.patch_dim,
            out_features=d_model
        )

    def image_to_patches(self, img, plot_figure=False):
        """ Split image into patches """
        x = img.unsqueeze(0)
        x = self.patch_embeddings(x)
        batch_size, number_of_channels, _ = x.shape
        x = x.reshape(
            batch_size, number_of_channels, self.patch_size, self.patch_size, self.channels
        ).squeeze(0)

        if not plot_figure:
            return x

        fig = plt.figure(figsize=(8, 8))
        matrix_shape = np.sqrt(number_of_channels).astype(int)

        for i in range(number_of_channels):
            patch = x[i]
            ax = fig.add_subplot(matrix_shape, matrix_shape, i + 1)
            ax.axes.get_xaxis().set_visible(False)
            ax.axes.get_yaxis().set_visible(False)
            ax.imshow(patch)

    def forward(self, x):
        x = self.patch_embeddings(x)
        x = self.embedding(x)
        return x

In [ ]:
# Refer to these links for more information about positional encoding 
# [1] https://kazemnejad.com/blog/transformer_architecture_positional_encoding/
# [2] https://machinelearningmastery.com/a-gentle-introduction-to-positional-encoding-in-transformer-models-part-1/ (code from here most likely won't work in out setting)
# [3] https://towardsdatascience.com/master-positional-encoding-part-i-63c05d90a0c3

class PositionalEncoding(torch.nn.Module):
    def __init__(self, d_model: int, max_seq_len: int = 256, dropout: float = 0.1):
        """
          Positional encoding class, that gives the model a notion of a position of a given patch representation
          Args:
            d_model: int 
              Embedding dims of the model 
            max_seq_len: int 
              maximum length
            dropout: float
              dropout rate
        """
        super().__init__()
        
        positions = torch.arange(0, max_seq_len).unsqueeze(1)
        div_factor = 10000 ** (torch.arange(0, d_model, 2) / d_model)

        positional_encoding = torch.zeros(max_seq_len, d_model)

        positional_encoding[:, 0::2] = torch.sin(positions / div_factor)
        positional_encoding[:, 1::2] = torch.cos(positions / div_factor)

        self.register_buffer('positional_encoding', positional_encoding)

        self.dropout = nn.Dropout(p=dropout)

    def visualize_positional_encoding(self):
        plt.figure(figsize=(8, 6))
        plt.imshow(self.positional_encoding, cmap='hot', interpolation='nearest')
        plt.show()

    def forward(self, x):
        """
        Args: 
            x: Tensor, shape [batch_size, seq_len, embedding_dim]
        """
        x = x + self.positional_encoding[:x.shape[1]]
        return self.dropout(x)


In [ ]:
# Useful links
# [1] https://stanford.edu/~shervine/teaching/cs-230/cheatsheet-recurrent-neural-networks
# [2] https://codeburst.io/recurrent-neural-network-4ca9fd4f242

class RNN(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        self.hidden_dim = hidden_dim

        self.W_xh = nn.Linear(input_dim, hidden_dim)
        self.W_hh = nn.Linear(hidden_dim, hidden_dim)
        self.b = nn.Parameter(torch.randn(hidden_dim))

    def step_forward(self, x, state):
        next_state = torch.tanh(self.W_xh(x) + self.W_hh(state) + self.b)
        return next_state

    def forward(self, inputs, state=None):
        b, t, _ = inputs.shape

        if state is None:
            state = torch.zeros(b, self.hidden_dim, device=inputs.device)

        outputs = torch.zeros((b, t, self.hidden_dim), device=inputs.device)

        for idx in range(t):
            state = self.step_forward(inputs[:, idx, :], state)
            outputs[:, idx, :] = state

        return outputs, state

In [ ]:
class Linear(nn.Module):
    def __init__(self, in_features, out_features, add_bias=True):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.add_bias = add_bias

        self.W = nn.Parameter(torch.randn(in_features, out_features))
        self.b = nn.Parameter(torch.randn(out_features)) if add_bias else None

    def forward(self, x):
        x = x @ self.W
        if self.add_bias and self.b is not None:
            x = x + self.b
        return x

In [ ]:
# Define your model here. 
# Initialize the RNN layer with hidden state dimension size 
# Initialize linear classification head. Make sure your output size matches the number of classes in the dataset

class MyModel(nn.Module):
    def __init__(self, patch_size, embedding_dim, recurrent_module: nn.Module, class_number: int, num_layers: int = 1, hidden_size: int = None):
        super().__init__()
        
        self.num_layers = num_layers
        self.hidden_size = hidden_size if hidden_size is not None else embedding_dim

        self.patch_embedding = PatchEmbeddings(
            patch_size=patch_size, d_model=embedding_dim
        )
        self.positional_encoding = PositionalEncoding(d_model=embedding_dim)
        
        self.recurrent_layers = nn.ModuleList()
        for i in range(num_layers):
            input_dim = embedding_dim if i == 0 else self.hidden_size
            self.recurrent_layers.append(recurrent_module(input_dim, self.hidden_size))
        
        self.classifier = nn.Linear(self.hidden_size, class_number)

    def forward(self, x):
        x = self.patch_embedding(x)
        x = self.positional_encoding(x)

        for rnn_layer in self.recurrent_layers:
            x, _ = rnn_layer(x)
        
        logits = self.classifier(x[:, -1, :])

        return logits

# Train loops and evaluation

Here you will need to implement training and evaluation methods, as in the previous homework

1) Define methods train, evaluation, and train_epoch

2) Define your model instance, loss function, optimizer, and metric function

3) Train model

4) Evaluate model

5) Do a small hyperparameter search and visualize your results

In [ ]:
def train_epoch(model: nn.Module, train_loader: torch.utils.data.DataLoader, criterion: nn.Module, optimizer: torch.optim.Optimizer, device: torch.device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for data, target in train_loader:
        data = data.float().to(device)
        target = target.long().to(device)

        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        pred = output.argmax(dim=1)
        correct += pred.eq(target).sum().item()
        total += target.size(0)
    
    return total_loss / len(train_loader), 100. * correct / total

def valid_epoch(model: nn.Module, valid_loader: torch.utils.data.DataLoader, criterion: nn.Module, device: torch.device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for data, target in valid_loader:
            data = data.float().to(device)
            target = target.long().to(device)

            output = model(data)
            loss = criterion(output, target)
            
            total_loss += loss.item()
            pred = output.argmax(dim=1)
            correct += pred.eq(target).sum().item()
            total += target.size(0)

    return total_loss / len(valid_loader), 100. * correct / total

def evaluate(model: nn.Module, test_loader: torch.utils.data.DataLoader, size: Tuple[int, int], device: torch.device, class_dict: dict):
    model.eval()
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for data, target in test_loader:
            data = data.float().to(device)
            target = target.long().to(device)

            output = model(data)

            pred = output.argmax(dim=1)
            
            all_preds.extend(pred.cpu().numpy())
            all_targets.extend(target.cpu().numpy())

    all_preds = np.array(all_preds)
    all_targets = np.array(all_targets)

    accuracy = (all_preds == all_targets).mean()
    precision_weighted = precision_score(all_targets, all_preds, average='weighted', zero_division=0)
    recall_weighted = recall_score(all_targets, all_preds, average='weighted', zero_division=0)
    f1_weighted = f1_score(all_targets, all_preds, average='weighted', zero_division=0)
    conf_matrix = confusion_matrix(all_targets, all_preds)

    print(f"Accuracy:  {accuracy:.4f}")
    print(f"Precision: {precision_weighted:.4f}")
    print(f"Recall:    {recall_weighted:.4f}")
    print(f"F1-Score:  {f1_weighted:.4f}")

    classes = [name for name, idx in sorted(class_dict.items(), key=lambda x: x[1])]
        
    fig = go.Figure(data=go.Heatmap(
        z=conf_matrix,
        x=classes,
        y=classes,
        colorscale='Blues',
        text=conf_matrix,
        texttemplate='%{text}',
        textfont={"size": 12},
        colorbar=dict(title="Count")
    ))
    
    fig.update_layout(
        title='Confusion Matrix',
        xaxis_title='Predicted Label',
        yaxis_title='True Label',
        width=700,
        height=700
    )
    
    fig.show()

def plot_best_trial_metrics(experiment_path):
    df = pd.read_csv(experiment_path)

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=('Loss', 'Accuracy')
    )

    fig.add_trace(
        go.Scatter(x=df['training_iteration'], y=df['train_loss'], 
                   mode='lines+markers', name='Train Loss'),
        row=1, col=1
    )

    fig.add_trace(
        go.Scatter(x=df['training_iteration'], y=df['loss'], 
                   mode='lines+markers', name='Test Loss'),
        row=1, col=1
    )

    fig.add_trace(
        go.Scatter(x=df['training_iteration'], y=df['train_accuracy'], 
                   mode='lines+markers', name='Train Accuracy'),
        row=1, col=2
    )

    fig.add_trace(
        go.Scatter(x=df['training_iteration'], y=df['accuracy'], 
                   mode='lines+markers', name='Test Accuracy'),
        row=1, col=2
    )

    fig.update_xaxes(title_text="Iteration", row=1, col=1)
    fig.update_xaxes(title_text="Iteration", row=1, col=2)
    fig.update_yaxes(title_text="Loss", row=1, col=1)
    fig.update_yaxes(title_text="Accuracy", row=1, col=2)

    fig.update_layout(height=500, width=1200, template='plotly_white', 
                      hovermode='x unified')

    fig.show()

In [ ]:
def train(config):
    train_transform = torchvision.transforms.Compose(
        [   
            torchvision.transforms.RandomHorizontalFlip(p=0.5),
            torchvision.transforms.GaussianBlur(kernel_size=(5, 5), sigma=(5, 11)),
            torchvision.transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
            torchvision.transforms.Resize((224, 224)),
            torchvision.transforms.ToTensor(),
            torchvision.transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ]
    )

    test_transform = torchvision.transforms.Compose(
        [   
            torchvision.transforms.Resize((224, 224)),
            torchvision.transforms.ToTensor(),
            torchvision.transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ]
    )

    train_loader, valid_loader, test_loader, class_dict = load_data(config, train_transform, test_transform)

    model = MyModel(
        patch_size=config['patch_size'],
        embedding_dim=config['embedding_dim'],
        recurrent_module=config['recurrent_module'],
        class_number=257,
        num_layers=config['num_layers'],
        hidden_size=config['hidden_size']
    )

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    criterion = torch.nn.CrossEntropyLoss(weight=config['weights'].to(device))
    optimizer = torch.optim.Adam(model.parameters(), lr=config['lr'], weight_decay=config["weight_decay"])
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience = 2)

    epochs = config.get('epochs', 20)

    for epoch in range(epochs):
        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
        valid_loss, valid_acc = valid_epoch(model, valid_loader, criterion, device)

        scheduler.step(valid_loss)

        with tempfile.TemporaryDirectory() as temp_checkpoint_dir:
            checkpoint_path = os.path.join(temp_checkpoint_dir, "checkpoint.pth")
            torch.save(model.state_dict(), checkpoint_path)
            
            tune.report(
                {"loss": valid_loss, "accuracy": valid_acc, "train_loss": train_loss, "train_accuracy": train_acc},
                checkpoint=Checkpoint.from_directory(temp_checkpoint_dir)
            )

cfg = {
    "lr": tune.loguniform(1e-4, 1e-1),
    "weight_decay": tune.loguniform(1e-6, 1e-2),
    "batch_size": tune.choice([32, 64, 128]),
    "epochs": 20,
    "patch_size": tune.choice([32, 16, 8]),
    "embedding_dim": tune.choice([128, 256, 512]),
    "recurrent_module": tune.choice([RNN, nn.GRU, nn.LSTM]),
    "num_layers": tune.choice([1, 2, 3]),
    "hidden_size": tune.choice([128, 256, 512]),
    "weights": weights
}

In [ ]:
scheduler = ASHAScheduler(
    metric="loss",
    mode="min",
    max_t=20,
    grace_period=1,
    reduction_factor=4
)

tuner = tune.Tuner(
    tune.with_resources(
        train,
        resources={"cpu": 4, "gpu": 0.5}
    ),
    param_space=cfg,
    tune_config=tune.TuneConfig(
        scheduler=scheduler,
        num_samples=20
    )
)

results = tuner.fit()

results = results.get_best_result(metric="loss", mode="min")
print(f"\nBest trial config: {results.config}")
print(f"Best trial final validation accuracy: {results.metrics['accuracy']:.2f}%")
print(f"Best trial final validation loss: {results.metrics['loss']:.4f}")